# HW2 — Part 4: Warehouse Robot with Mixed Actions
### Actor-Critic · Categorical + Normal Distributions · Continuous Control
**Yogeshvar Reddy Kallam** · IST 597 Deep RL · Penn State Spring 2025

---

## Mixed Discrete-Continuous Action Space

Action = `(action_type: Discrete(3), movement: Continuous[-1,1])`

- `MOVE (0)`:    move by `movement + N(0,0.1)`, clipped to [0,3]
- `PICK_UP (1)`: if pos ∈ [0.5,1.5] and no package → grab
- `DROP_OFF (2)`: if carrying: +1 (pos ∈ [2,3]) else −0.1

**Policy Network:** shared encoder → Categorical head + Normal head  
**Baseline:** separate Value Network for advantage estimation

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical, Normal
import gymnasium as gym

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class WarehouseRobotMixedEnv(gym.Env):
    MOVE=0; PICK_UP=1; DROP_OFF=2; NO_PACKAGE=0; PACKAGE=1
    def __init__(self, max_steps=200):
        super().__init__()
        self.observation_space = gym.spaces.Box(low=np.array([0.,0.]),high=np.array([3.,1.]),dtype=np.float32)
        self.action_space = gym.spaces.Tuple((gym.spaces.Discrete(3),gym.spaces.Box(-1.,1.,(1,),np.float32)))
        self.max_steps=max_steps; self.rng=np.random.default_rng()
    def reset(self, seed=None, options=None):
        super().reset(seed=seed); self.state=np.array([1.,0.],dtype=np.float32); self.step_ct=0; return self.state.copy(),{}
    def step(self,action):
        at,mv=action; r=0.
        p,pkg=self.state
        if at==self.MOVE: self.state[0]=float(np.clip(p+float(np.clip(mv,-1.,1.))+self.rng.normal(0,.1),0.,3.))
        elif at==self.PICK_UP and .5<=p<=1.5 and pkg==0: self.state[1]=1
        elif at==self.DROP_OFF and pkg==1:
            self.state[1]=0; r=1. if 2.<=p<=3. else -0.1
        self.step_ct+=1; return self.state.copy(),r,False,self.step_ct>=self.max_steps,{}

class Policy(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1=nn.Linear(2,256); self.fc2=nn.Linear(256,256)
        self.logits=nn.Linear(256,3); self.mu=nn.Linear(256,1); self.logsig=nn.Linear(256,1)
    def forward(self,x):
        x=F.relu(self.fc1(x)); x=F.relu(self.fc2(x))
        return self.logits(x),self.mu(x),self.logsig(x)

class Value(nn.Module):
    def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Linear(2,256),nn.ReLU(),nn.Linear(256,256),nn.ReLU(),nn.Linear(256,1))
    def forward(self,x): return self.net(x)

env=WarehouseRobotMixedEnv(); pol=Policy().to(DEVICE); val=Value().to(DEVICE)
po=optim.Adam(pol.parameters(),lr=1e-4); vo=optim.Adam(val.parameters(),lr=1e-3)
rewards_hist=[]
for ep in range(1,1001):
    s,_=env.reset(); lps,rs,vs=[],[],[]
    done=trunc=False
    while not(done or trunc):
        st=torch.tensor(s,dtype=torch.float32,device=DEVICE)
        lg,mu,lsig=pol(st); cat=Categorical(logits=lg); at=cat.sample()
        sig=torch.exp(lsig.clamp(-4,2)); norm=Normal(mu,sig)
        if at.item()==0:
            rm=norm.sample(); mv=torch.tanh(rm).item()
            lp=cat.log_prob(at)+norm.log_prob(rm).sum()
        else: mv=0.; lp=cat.log_prob(at)
        ns,r,done,trunc,_=env.step((at.item(),mv))
        lps.append(lp); rs.append(r); vs.append(val(st)); s=ns
    G=0.; pl=0.; vl=0.
    for t in reversed(range(len(rs))):
        G=rs[t]+0.85*G; adv=G-vs[t].detach()
        pl+=-lps[t]*adv; vl+=(G-vs[t])**2
    po.zero_grad(); pl.backward(); po.step()
    vo.zero_grad(); vl.backward(); vo.step()
    rewards_hist.append(sum(rs))
    if ep%200==0: print(f"Ep {ep}: avg={np.mean(rewards_hist[-200:]):.3f}")
print("Training complete.")
